# Mirror Pair Validation

Validates the A/B replicate imaging protocol by checking whether mirror pairs
cluster closer in morphospace than chance. Each individual has a Region A and Region B
sample (mirror scalp locations). If the pipeline captures real biology, A and B from the
same individual should be closer to each other than to other individuals' samples.


## Setup

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path('../data')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

mv = pd.read_csv(DATA_DIR / 'mirror_validation.csv')
mv['dist_ratio'] = mv['pair_distance'] / mv['mean_non_pair_dist']
print(f'Mirror pairs: {len(mv)}')
print(f'Nearest-neighbour rate: {mv["is_nearest_neighbour"].mean()*100:.1f}%')
print(f'Compression ratio (mean pair_dist / mean non-pair dist): {mv["dist_ratio"].mean():.3f}')


## Interpreting compression ratio

A compression ratio < 1 means mirror pairs are systematically closer than a random pair.
A ratio of 0.44 means mirror pairs are ~2.3× closer than the population average —
direct evidence the expanded feature space captures real biological signal reproducible
across measurement replicates.


## NN rank distribution

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.hist(mv['nn_rank_of_B'], bins=40, color='steelblue', edgecolor='white', linewidth=0.3)
ax1.axvline(mv['nn_rank_of_B'].median(), color='crimson', linestyle='--',
            label=f'Median rank = {mv["nn_rank_of_B"].median():.0f}')
ax1.set_xlabel('NN rank of B in A\'s neighbourhood')
ax1.set_ylabel('Pairs')
ax1.set_title('NN rank distribution')
ax1.legend()

ax2.hist(mv['dist_ratio'], bins=40, color='#4C72B0', edgecolor='white', linewidth=0.3)
ax2.axvline(1.0, color='grey', linestyle=':', label='Random baseline (ratio=1)')
ax2.axvline(mv['dist_ratio'].mean(), color='crimson', linestyle='--',
            label=f'Mean ratio = {mv["dist_ratio"].mean():.3f}')
ax2.set_xlabel('pair_distance / mean_non_pair_dist')
ax2.set_ylabel('Pairs')
ax2.set_title('Compression ratio distribution')
ax2.legend()

plt.tight_layout(); plt.show()


## A–B scatter with pair lines

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
cmap = plt.cm.RdYlGn_r
norm = plt.Normalize(mv['nn_rank_of_B'].min(), mv['nn_rank_of_B'].quantile(0.95))

for _, row in mv.iterrows():
    color = cmap(norm(row['nn_rank_of_B']))
    ax.plot([row['pc1_A'], row['pc1_B']], [row['pc2_A'], row['pc2_B']],
            color=color, alpha=0.4, linewidth=0.6)

ax.scatter(mv['pc1_A'], mv['pc2_A'], s=8, c='steelblue', label='Region A', linewidths=0)
ax.scatter(mv['pc1_B'], mv['pc2_B'], s=8, c='darkorange', label='Region B', linewidths=0)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
plt.colorbar(sm, ax=ax, label='NN rank (green=1, red=high)')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title('A–B mirror pairs in PC1–PC2 space\n(lines connect mirror pairs, coloured by NN rank)')
ax.legend()
plt.tight_layout(); plt.show()


## Poor mirror pairs (dist_ratio > 1.0)

In [ ]:
poor = mv[mv['dist_ratio'] > 1.0]
print(f'Pairs where mirrors are farther than a random pair: {len(poor)} / {len(mv)} ({len(poor)/len(mv)*100:.1f}%)')
poor.sort_values('dist_ratio', ascending=False).head(10)[['pair_key','individual','replicate','pair_distance','nn_rank_of_B','dist_ratio']].round(3)
